# 07 · x402 — pay for AI with a Base wallet, no API key

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabrinaaquino/base-batches-workshop/blob/main/notebooks/07-x402-wallet-payments.ipynb)

**x402** is an HTTP payment standard. Instead of an API key, you authenticate with a signed message from your **Base wallet** and pay per request in **USDC on Base**.

By the end of this notebook you will have:
1. Generated a fresh wallet (or used your own)
2. Topped it up with USDC on Base via the x402 protocol
3. Made a Venice chat completion authenticated with **only a wallet signature** — no Venice account, no email, no API key

**You'll need:** a Base wallet with at least **$5 USDC** (the minimum top-up). New to Base? You can withdraw USDC directly from Coinbase (select "Base" as the network) to your wallet.

In [ ]:
%pip install --quiet openai requests python-dotenv rich web3 eth-account

## 1. Set up the wallet

Two options below — pick one.

**Option A: Use an existing wallet** — set `WALLET_PRIVATE_KEY` as a Colab secret (or env var) and run the cell.

**Option B: Generate a fresh wallet** — uncomment the second cell. You'll then need to send it some USDC on Base before continuing.

In [ ]:
import os
from eth_account import Account

# Option A: load from env / Colab secrets
try:
    from google.colab import userdata  # type: ignore
    pk = userdata.get("WALLET_PRIVATE_KEY")
except Exception:
    pk = os.environ.get("WALLET_PRIVATE_KEY")

if not pk:
    from getpass import getpass
    pk = getpass("Paste your Base wallet private key (or skip and run the next cell to generate one): ").strip()

if pk:
    if not pk.startswith("0x"):
        pk = "0x" + pk
    account = Account.from_key(pk)
    print(f"Wallet address: {account.address}")
else:
    account = None
    print("No key provided. Run the next cell to generate one.")

In [ ]:
# Option B (uncomment to generate a fresh wallet)

# from eth_account import Account
# acc = Account.create()
# print(f"Address:     {acc.address}")
# print(f"Private key: {acc.key.hex()}")
# print()
# print("Send at least $5 USDC on Base to that address before continuing.")
# print("USDC contract on Base: 0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913")

## 2. Check your USDC balance on Base

In [ ]:
from web3 import Web3

w3 = Web3(Web3.HTTPProvider("https://mainnet.base.org"))
USDC = Web3.to_checksum_address("0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913")

erc20_abi = [{
    "constant": True,
    "inputs": [{"name": "_owner", "type": "address"}],
    "name": "balanceOf",
    "outputs": [{"name": "balance", "type": "uint256"}],
    "type": "function",
}]
contract = w3.eth.contract(address=USDC, abi=erc20_abi)
raw = contract.functions.balanceOf(account.address).call()
print(f"USDC balance on Base: ${raw / 10**6:,.2f}")

## 3. Use the official Venice x402 client

We'll use the Venice-published Python helper. It handles the SIWE signing, the 402 round-trip, the EIP-3009 USDC authorization, and balance tracking for you.

In [ ]:
%pip install --quiet venice-x402-client

In [ ]:
# Note: at the time of writing the official Venice x402 client ships as a
# TypeScript package. If a Python port is not yet on PyPI, the cells below
# show the raw HTTP flow that the SDK wraps. Both work — the SDK is just sugar.

import requests, base64, json, secrets, time
from eth_account.messages import encode_defunct


VENICE = "https://api.venice.ai/api/v1"


def make_siwe_header(account, resource_url: str) -> str:
    """Build the X-Sign-In-With-X header (base64-encoded SIWE message)."""
    nonce = secrets.token_hex(16)
    issued_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

    message = (
        f"venice.ai wants you to sign in with your Ethereum account:\n"
        f"{account.address}\n\n"
        f"Sign in to Venice AI\n\n"
        f"URI: {resource_url}\n"
        f"Version: 1\n"
        f"Chain ID: 8453\n"
        f"Nonce: {nonce}\n"
        f"Issued At: {issued_at}"
    )

    sig = account.sign_message(encode_defunct(text=message)).signature.hex()

    payload = {
        "message": message,
        "signature": sig if sig.startswith("0x") else f"0x{sig}",
        "address": account.address,
    }
    return base64.b64encode(json.dumps(payload).encode()).decode()


# Test the auth: hitting the balance endpoint requires only a signature, no payment
header = make_siwe_header(account, f"{VENICE}/x402/balance/{account.address}")
r = requests.get(
    f"{VENICE}/x402/balance/{account.address}",
    headers={"X-Sign-In-With-X": header},
)
print(r.status_code, r.json())

## 4. Top up

Hit `/x402/top-up` once with no payment header — Venice replies with `402 Payment Required` and a JSON envelope describing what to sign. Sign an EIP-3009 `TransferWithAuthorization` for that amount of USDC on Base, then re-submit with the signed payment in the `X-402-Payment` header.

The Venice x402 SDK handles all of this in a single `topUp()` call. Below is the raw shape so you can see what's happening under the hood.

In [ ]:
# Step 1: ask for payment requirements
header = make_siwe_header(account, f"{VENICE}/x402/top-up")
r = requests.post(
    f"{VENICE}/x402/top-up",
    headers={"X-Sign-In-With-X": header},
)
print(f"Status: {r.status_code}")
print("Payment requirements:")
print(json.dumps(r.json(), indent=2)[:800])

The full top-up flow (signing the EIP-3009 authorization, settling on Base, polling for confirmation) is ~80 lines. For the workshop we recommend using the maintained SDK rather than rolling your own — see the [Venice x402 docs](https://docs.venice.ai/overview/guides/x402-venice-api) for current Python examples.

## 5. Make a paid call

Once you have a balance, you can hit any inference endpoint with the SIWE header instead of an API key:

In [ ]:
header = make_siwe_header(account, f"{VENICE}/chat/completions")

r = requests.post(
    f"{VENICE}/chat/completions",
    headers={
        "X-Sign-In-With-X": header,
        "Content-Type": "application/json",
    },
    json={
        "model": "venice-uncensored",
        "messages": [{"role": "user", "content": "Say hi from a fresh wallet."}],
    },
)

if r.status_code == 200:
    print(r.json()["choices"][0]["message"]["content"])
    print()
    print(f"Balance remaining: ${r.headers.get('x-balance-remaining', '?')}")
else:
    print(r.status_code, r.text[:500])

## What just happened

A wallet that didn't exist this morning **just bought itself an AI subscription**. No signup, no API key, no email.

This unlocks autonomous agents that can pay for their own inference, and tools that don't have to manage user accounts at all — they just charge per call.

**Next:** [08 · E2EE — the cryptography](./08-e2ee-encryption.ipynb)